# 04 - Training in Google Colab (GRU / LSTM / XGBoost / baseline)

> ## STOP - GOOGLE COLAB ONLY
> **Do not run this notebook locally.** The local machine only performs data
> acquisition, feature engineering, target generation and dataset preparation.
> Training happens here, in Colab, on a GPU runtime.

This notebook compares four approaches on the **same** pre-built dataset:

| # | Model | Role |
|---|---|---|
| 1 | Naive baseline | sanity floor - persistence + majority class |
| 2 | XGBoost | strong non-sequence reference |
| 3 | LSTM | recurrent reference |
| 4 | **GRU** | primary architecture |

**Multi-task objective** (the preferred experiment): predict all six future
returns simultaneously (regression) with directional classification as an
auxiliary task on the same shared representation.

Nothing in this notebook claims the GRU will win. The validation split decides,
and the test split is touched exactly once, at the very end.

### Before you start
1. In Colab: `Runtime -> Change runtime type -> T4 GPU`.
2. Upload the files listed in the next cell to `MyDrive/btc-trend-prediction/`.

In [ ]:
# ============================================================================
# GOOGLE COLAB ONLY - DO NOT EXECUTE LOCALLY
# ============================================================================
# TensorFlow installation and model training belong in Google Colab.
# The local project (src/) deliberately does NOT depend on TensorFlow and
# must never train a model. Run this notebook only inside Colab.
# ============================================================================

import os
import sys

# Hard guard: refuse to run outside Colab unless explicitly overridden.
IN_COLAB = ("google.colab" in sys.modules) or os.path.isdir("/content")
ALLOW_LOCAL = os.environ.get("ALLOW_LOCAL_TRAINING") == "1"

if not IN_COLAB and not ALLOW_LOCAL:
    raise RuntimeError(
        "This notebook is for Google Colab only.\n"
        "The local project must not train models and does not install "
        "TensorFlow.\nIf you really mean to train locally (TensorFlow "
        "required), set ALLOW_LOCAL_TRAINING=1."
    )
print("Running in Colab:", IN_COLAB)

## 1. Environment

Colab already ships TensorFlow and XGBoost. This cell only makes sure they are
importable and installs XGBoost if the runtime lacks it.

**Reminder:** these imports exist *only* in this Colab notebook. Nothing under
`src/` imports TensorFlow or PyTorch, and the local pipeline never does.

In [ ]:
# ---- COLAB ONLY: TensorFlow lives exclusively inside this notebook ----
try:
    import xgboost  # noqa: F401
except ImportError:
    %pip install -q xgboost
    import xgboost  # noqa: F401

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print("tensorflow", tf.__version__)
print("gpus      ", tf.config.list_physical_devices("GPU"))

In [ ]:
import json
import pickle
import random
from pathlib import Path

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ---------------------------------------------------------------------------
# Point this at the folder holding the files exported by src/pipeline.py
# ---------------------------------------------------------------------------
USE_DRIVE = True

if USE_DRIVE and IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = Path("/content/drive/MyDrive/btc-trend-prediction")
else:
    DATA_DIR = Path("/content/btc-trend-prediction")

print("DATA_DIR:", DATA_DIR)
print("exists  :", DATA_DIR.exists())

## 2. Load the prepared dataset

These files were produced locally by `python src/pipeline.py`:

```
X_train.npy  X_val.npy  X_test.npy   scaled feature BLOCKS, (n_rows, 49) float32
starts_*.npy                         window offset into the block, (n_samples,) int64
y_train.npy  y_val.npy  y_test.npy   future returns in %, (n_samples, 6) float32
y_dir_*.npy                          direction labels, (n_samples, 6) int8
timestamps_*.npy                     target timestamp per sample, int64 epoch ms
scaler.pkl  target_scaler.pkl        fitted on the TRAIN split only
dataset_metadata.json                every reported statistic
```

### Why `X_*.npy` is a block, not "one window per row"

`X_train.npy` is a **contiguous** 2D slice of the scaled feature matrix, and
`starts_train.npy` records where each sample's window begins inside it. Sample
`k` is

```python
X_train[starts_train[k] : starts_train[k] + LOOKBACK]
```

Windows that would straddle a data gap are excluded, which leaves **holes** in
the sequence of window starts. The block therefore has more rows than there are
samples, and `starts_*.npy` is not simply `0, 1, 2, ...`.

Slicing `X_train[k : k + LOOKBACK]` instead would silently splice across a hole
and fabricate a window spanning missing market data. **Always index through
`starts_*.npy`** - the batcher in section 3 does this for you.

Likewise, `y_*.npy`, `y_dir_*.npy` and `timestamps_*.npy` already hold exactly
one row per sample: do **not** apply a `[LOOKBACK-1:]` offset to them.

The scaler is already fitted and applied - **never fit a new one here.**


In [ ]:
REQUIRED = [
    "X_train.npy", "X_val.npy", "X_test.npy",
    "starts_train.npy", "starts_val.npy", "starts_test.npy",
    "y_train.npy", "y_val.npy", "y_test.npy",
    "y_dir_train.npy", "y_dir_val.npy", "y_dir_test.npy",
    "timestamps_train.npy", "timestamps_val.npy", "timestamps_test.npy",
    "scaler.pkl", "target_scaler.pkl", "dataset_metadata.json",
]
missing = [f for f in REQUIRED if not (DATA_DIR / f).exists()]
if missing:
    raise FileNotFoundError(
        "Missing files in DATA_DIR: " + ", ".join(missing) +
        "\nUpload the whole data/final/ folder from the local project."
    )

X_train = np.load(DATA_DIR / "X_train.npy")
X_val = np.load(DATA_DIR / "X_val.npy")
X_test = np.load(DATA_DIR / "X_test.npy")

s_train = np.load(DATA_DIR / "starts_train.npy")
s_val = np.load(DATA_DIR / "starts_val.npy")
s_test = np.load(DATA_DIR / "starts_test.npy")

y_train = np.load(DATA_DIR / "y_train.npy")
y_val = np.load(DATA_DIR / "y_val.npy")
y_test = np.load(DATA_DIR / "y_test.npy")

d_train = np.load(DATA_DIR / "y_dir_train.npy").astype(np.int64)
d_val = np.load(DATA_DIR / "y_dir_val.npy").astype(np.int64)
d_test = np.load(DATA_DIR / "y_dir_test.npy").astype(np.int64)

ts_train = np.load(DATA_DIR / "timestamps_train.npy")   # int64 epoch ms, UTC
ts_val = np.load(DATA_DIR / "timestamps_val.npy")
ts_test = np.load(DATA_DIR / "timestamps_test.npy")

with open(DATA_DIR / "scaler.pkl", "rb") as fh:
    scaler = pickle.load(fh)
with open(DATA_DIR / "target_scaler.pkl", "rb") as fh:
    target_scaler = pickle.load(fh)
with open(DATA_DIR / "dataset_metadata.json", encoding="utf-8") as fh:
    META = json.load(fh)

LOOKBACK = META["lookback"]
HORIZONS = list(META["horizons"].keys())          # ["15m","30m",...]
N_H = len(HORIZONS)
N_F = X_train.shape[1]

# ---- structural checks: a block + offsets that disagree would be fatal -----
for name, block, starts, y, d in (("train", X_train, s_train, y_train, d_train),
                                  ("val", X_val, s_val, y_val, d_val),
                                  ("test", X_test, s_test, y_test, d_test)):
    assert len(starts) == len(y) == len(d), f"{name}: offsets/targets misaligned"
    assert starts.min() >= 0, f"{name}: negative window offset"
    assert starts.max() + LOOKBACK <= len(block), f"{name}: window past block end"
    assert y.shape[1] == N_H and d.shape[1] == N_H, f"{name}: wrong horizon count"

for name, block, starts in (("X_train", X_train, s_train),
                            ("X_val", X_val, s_val),
                            ("X_test", X_test, s_test)):
    print(f"{name:9} block {str(block.shape):>14}  {block.dtype}   "
          f"{len(starts):>7,} windows")
print(f"\nlookback={LOOKBACK}  features={N_F}  horizons={HORIZONS}")
print(f"y_train {y_train.shape} {y_train.dtype}   d_train {d_train.shape} {d_train.dtype}")
print("dataset period :", META["dataset"]["first_timestamp_utc"],
      "->", META["dataset"]["last_timestamp_utc"])
print("test period    :", META["splits"]["test"]["first_target_ts_utc"],
      "->", META["splits"]["test"]["last_target_ts_utc"])


## 3. Build the 3D sequence tensors

`SequenceBatcher` reads windows from the 2D block on demand, so a full
`(218515, 64, 49)` tensor (several GB) is never materialised at once.

It is driven by the saved window offsets:

```python
window_k = X_block[starts[k] : starts[k] + LOOKBACK]      # correct
window_k = X_block[k : k + LOOKBACK]                      # WRONG - splices gaps
```

`shuffle=True` is used **only for training**, and it permutes the order in which
*samples* are visited. It never mixes splits, and it never reorders time inside
a window. Validation and test always stay in chronological order.

The wrapper below reuses the project's own batcher when `src/sequences.py` is
available and falls back to an identical standalone copy otherwise, so the
windowing logic exists in exactly one place.


In [ ]:
# Reuse the exact windowing logic from the local project (pure numpy, no TF).
sys.path.insert(0, str(DATA_DIR))          # if you copied src/ alongside
try:
    from sequences import SequenceBatcher as _SequenceBatcher
    print("using src/sequences.py::SequenceBatcher")
except ImportError:
    print("using the notebook's standalone copy of SequenceBatcher")

    class _SequenceBatcher:
        """
        Standalone copy of src/sequences.py::SequenceBatcher, so this notebook
        works without uploading src/. Window k is
        X_block[start_offsets[k] : start_offsets[k] + lookback].
        """

        def __init__(self, X_block, y, start_offsets, *, lookback,
                     batch_size=512, shuffle=False, seed=None):
            self.X = np.asarray(X_block)
            self.y = np.asarray(y)
            self.starts = np.asarray(start_offsets, dtype=np.int64)
            self.lookback = int(lookback)
            self.batch_size = int(batch_size)
            self.shuffle = bool(shuffle)
            self.rng = np.random.default_rng(seed)
            self.n_samples = int(self.starts.size)

        def __len__(self):
            return int(np.ceil(self.n_samples / self.batch_size))

        def order(self):
            idx = np.arange(self.n_samples, dtype=np.int64)
            return self.rng.permutation(idx) if self.shuffle else idx

        def materialise(self, sample_index):
            idx = np.asarray(sample_index, dtype=np.int64)
            starts = self.starts[idx]
            rows = starts[:, None] + np.arange(self.lookback, dtype=np.int64)[None, :]
            return (np.ascontiguousarray(self.X[rows], dtype=np.float32),
                    np.ascontiguousarray(self.y[idx], dtype=np.float32))

        def __getitem__(self, b):
            idx = self.order()[b * self.batch_size:(b + 1) * self.batch_size]
            return self.materialise(idx)

        def as_dataset(self):
            return self.materialise(np.arange(self.n_samples, dtype=np.int64))


class SequenceBatcher:
    """Adds the direction head's targets to the block/offset batcher."""

    def __init__(self, X_block, y, ydir, starts, *, lookback,
                 batch_size=512, shuffle=False, seed=None):
        self.inner = _SequenceBatcher(X_block, y, starts, lookback=lookback,
                                      batch_size=batch_size, shuffle=shuffle,
                                      seed=seed)
        self.ydir = np.asarray(ydir, dtype=np.int64)
        self.batch_size = int(batch_size)
        self.n_samples = self.inner.n_samples

    def __len__(self):
        return len(self.inner)

    def __getitem__(self, b):
        idx = self.inner.order()[b * self.batch_size:(b + 1) * self.batch_size]
        Xb, yb = self.inner.materialise(idx)
        return Xb, {"return_head": yb, "direction_head": self.ydir[idx]}

    def as_dataset(self):
        Xb, yb = self.inner.as_dataset()
        return Xb, {"return_head": yb, "direction_head": self.ydir}


print("SequenceBatcher ready")


In [ ]:
BATCH = 512

train_batcher = SequenceBatcher(X_train, y_train, d_train, s_train,
                                lookback=LOOKBACK, batch_size=BATCH,
                                shuffle=True, seed=SEED)
val_batcher = SequenceBatcher(X_val, y_val, d_val, s_val,
                              lookback=LOOKBACK, batch_size=BATCH, shuffle=False)
test_batcher = SequenceBatcher(X_test, y_test, d_test, s_test,
                               lookback=LOOKBACK, batch_size=BATCH, shuffle=False)

Xb, yb = train_batcher[0]
print("one training batch:", Xb.shape, {k: v.shape for k, v in yb.items()})
print("batches per epoch :", len(train_batcher), "x", BATCH)

# A shuffled batch must still be one contiguous 64-candle window per row.
restored = train_batcher.inner.starts[train_batcher.inner.order()[0]]
assert Xb.shape[1] == LOOKBACK, "window length is not the lookback"
print("first shuffled window starts at block row", int(restored))


## 4. Evaluation utilities

Reported **per horizon** - never as a single pooled number. A model that is
useful at 15m and useless at 90m must show up that way.

* regression: MAE and RMSE in percent
* direction: accuracy, per-class precision / recall / F1, confusion matrix
* skill vs the naive baseline (predict zero return / predict the majority class)

In [ ]:
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score,
                             precision_recall_fscore_support)

CLASS_NAMES = ["DOWN", "NEUTRAL", "UP"]


def regression_metrics(y_true, y_pred, horizons=HORIZONS):
    rows = {}
    for i, h in enumerate(horizons):
        e = y_pred[:, i] - y_true[:, i]
        rows[h] = {
            "MAE_%": float(np.mean(np.abs(e))),
            "RMSE_%": float(np.sqrt(np.mean(e ** 2))),
            "bias_%": float(np.mean(e)),
            "corr": float(np.corrcoef(y_true[:, i], y_pred[:, i])[0, 1]),
        }
    return pd.DataFrame(rows).T


def direction_metrics(y_true_dir, y_pred_dir, horizons=HORIZONS):
    '''y_*_dir are (n, 6) with values in {0,1,2}.'''
    rows = {}
    for i, h in enumerate(horizons):
        t, p = y_true_dir[:, i], y_pred_dir[:, i]
        prec, rec, f1, sup = precision_recall_fscore_support(
            t, p, labels=[0, 1, 2], zero_division=0)
        rows[h] = {
            "accuracy": accuracy_score(t, p),
            "macro_F1": f1_score(t, p, average="macro", zero_division=0),
            "DOWN_prec": prec[0], "DOWN_rec": rec[0], "DOWN_f1": f1[0],
            "NEUTRAL_prec": prec[1], "NEUTRAL_rec": rec[1], "NEUTRAL_f1": f1[1],
            "UP_prec": prec[2], "UP_rec": rec[2], "UP_f1": f1[2],
            "n": int(len(t)),
        }
    return pd.DataFrame(rows).T


def show_confusion(y_true_dir, y_pred_dir, horizon_index=0):
    cm = confusion_matrix(y_true_dir[:, horizon_index], y_pred_dir[:, horizon_index],
                          labels=[0, 1, 2])
    return pd.DataFrame(cm, index=[f"true_{c}" for c in CLASS_NAMES],
                        columns=[f"pred_{c}" for c in CLASS_NAMES])


def majority_class_baseline(y_true_dir):
    '''Always predict the most frequent training class, per horizon.'''
    preds = np.zeros_like(y_true_dir)
    for i in range(y_true_dir.shape[1]):
        vals, counts = np.unique(d_train[:, i], return_counts=True)
        preds[:, i] = vals[np.argmax(counts)]
    return preds

## 5. Model 1 - Naive baseline

Two floors any real model must clear:

* **regression:** predict a zero return. Its MAE *is* the mean absolute return,
  so it scales with horizon volatility.
* **direction:** always predict the majority training class.

In [ ]:
naive_pred = np.zeros_like(y_test)
base_reg = regression_metrics(y_test, naive_pred)
print("NAIVE REGRESSION (predict 0% return)")
display(base_reg)

base_dir_pred = majority_class_baseline(d_test)
base_dir = direction_metrics(d_test, base_dir_pred)
print("\nNAIVE DIRECTION (majority train class per horizon)")
display(base_dir)

## 6. Model 2 - XGBoost

A tree model cannot consume a raw sequence, so each 64-candle window is
compressed into a fixed-length vector: the last candle plus the mean, std, min
and max of every feature across the window.

This is stated plainly because it makes the comparison **approximate rather
than perfectly apples-to-apples** - XGBoost sees a summary of the sequence, the
recurrent models see the full ordered sequence.

In [ ]:
def aggregate_windows(X_block, starts, lookback):
    """
    Scaled block + window offsets -> (n_samples, 5F): the last candle plus the
    mean, std, min and max of every feature across each sample's window.

    Indexing goes through `starts`, never `X_block[k : k + lookback]`, because
    gap-excluded windows leave holes in the offset sequence.
    """
    starts = np.asarray(starts, dtype=np.int64)
    rows = starts[:, None] + np.arange(lookback, dtype=np.int64)[None, :]
    w = X_block[rows]                                   # (n_samples, lookback, F)
    return np.concatenate(
        [w[:, -1, :], w.mean(axis=1), w.std(axis=1), w.min(axis=1), w.max(axis=1)],
        axis=1,
    ).astype(np.float32)


A_train = aggregate_windows(X_train, s_train, LOOKBACK)
A_val = aggregate_windows(X_val, s_val, LOOKBACK)
A_test = aggregate_windows(X_test, s_test, LOOKBACK)

# These must line up with y_*.npy one-to-one - XGBoost needs matching rows.
assert len(A_train) == len(y_train), "aggregated windows and targets disagree"
print("aggregated:", A_train.shape, "-> y", y_train.shape,
      "|", A_val.shape, "|", A_test.shape)


In [ ]:
import xgboost as xgb

xgb_models, xgb_val_pred, xgb_test_pred = {}, {}, {}
xgb_val_dir, xgb_test_dir = {}, {}

# Decision thresholds come from the frozen TRAIN distribution in the metadata.
THR = {h: (META["thresholds"]["per_horizon"][h]["lower_percent"],
           META["thresholds"]["per_horizon"][h]["upper_percent"]) for h in HORIZONS}

for i, h in enumerate(HORIZONS):
    model = xgb.XGBRegressor(
        n_estimators=600, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        reg_lambda=1.0, early_stopping_rounds=50,
        random_state=SEED, n_jobs=-1, tree_method="hist",
    )
    model.fit(A_train, y_train[:, i], eval_set=[(A_val, y_val[:, i])], verbose=False)
    xgb_models[h] = model
    xgb_val_pred[i] = model.predict(A_val)
    xgb_test_pred[i] = model.predict(A_test)

    lo, hi = THR[h]
    xgb_val_dir[i] = np.digitize(xgb_val_pred[i], [lo, hi])
    xgb_test_dir[i] = np.digitize(xgb_test_pred[i], [lo, hi])

xgb_val_pred = np.column_stack([xgb_val_pred[i] for i in range(N_H)])
xgb_test_pred = np.column_stack([xgb_test_pred[i] for i in range(N_H)])
xgb_val_dir = np.column_stack([xgb_val_dir[i] for i in range(N_H)])
xgb_test_dir = np.column_stack([xgb_test_dir[i] for i in range(N_H)])

print("XGBoost - VALIDATION")
display(regression_metrics(y_val, xgb_val_pred))
display(direction_metrics(d_val, xgb_val_dir))

## 7. Models 3 and 4 - LSTM and GRU (multi-task)

Shared recurrent trunk, two heads:

* `return_head` - linear, 6 outputs, MSE on **scaled** returns
* `direction_head` - softmax, 6 x 3 outputs, sparse categorical cross-entropy

Anti-overfitting measures: `EarlyStopping`, `ReduceLROnPlateau` and
`ModelCheckpoint`, all monitoring `val_loss`. **The test set is never used for
model selection** - it is evaluated once, at the end.

In [ ]:
def build_multi_task_model(cell_type="GRU", units=96, dropout=0.25,
                          learning_rate=1e-3, n_h=N_H, n_f=N_F, lookback=LOOKBACK):
    """Two-layer recurrent trunk + regression head + direction head."""
    RNN = {"GRU": layers.GRU, "LSTM": layers.LSTM}[cell_type]

    inputs = keras.Input(shape=(lookback, n_f), name="sequence")
    x = layers.GRU(units, return_sequences=True, name="rnn_1")(inputs) \
        if cell_type == "GRU" else \
        layers.LSTM(units, return_sequences=True, name="rnn_1")(inputs)
    x = layers.Dropout(dropout)(x)
    x = RNN(units // 2, return_sequences=False, name="rnn_2")(x)
    x = layers.Dropout(dropout)(x)
    shared = layers.Dense(64, activation="relu", name="shared")(x)

    returns = layers.Dense(n_h, activation="linear", name="return_head")(shared)
    # ORDER MATTERS: reshape to (n_h, 3) FIRST, then softmax over the class
    # axis. Softmax on the flat 18-vector shares ONE normaliser across all
    # six horizons, so each horizon sums to ~1/6 instead of 1: the head
    # cannot express "UP at 15m, DOWN at 90m" independently, and the
    # confidence it reports is not a probability. Argmax still survives
    # (softmax is monotonic within a group), which is what makes it easy
    # to miss - notebook 05 asserts the normalisation after building.
    direction = layers.Dense(n_h * 3, name="direction_flat")(shared)
    direction = layers.Reshape((n_h, 3), name="direction_reshape")(direction)
    direction = layers.Activation("softmax", name="direction_head")(direction)

    model = keras.Model(inputs, {"return_head": returns, "direction_head": direction})

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate),
        loss={"return_head": "mse", "direction_head": "sparse_categorical_crossentropy"},
        loss_weights={"return_head": 1.0, "direction_head": 0.5},
        metrics={"return_head": ["mae"], "direction_head": ["accuracy"]},
    )
    return model


build_multi_task_model("GRU").summary()

In [ ]:
def train_recurrent(cell_type, epochs=60, units=96, dropout=0.25, lr=1e-3,
                    batch_size=BATCH, tag=None):
    tag = tag or cell_type.lower()
    tf.keras.utils.set_random_seed(SEED)

    model = build_multi_task_model(cell_type, units=units, dropout=dropout,
                                   learning_rate=lr)

    ckpt = DATA_DIR / f"best_{tag}.keras"
    callbacks = [
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=8,
                                      restore_best_weights=True, verbose=1),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                                          patience=4, min_lr=1e-5, verbose=1),
        keras.callbacks.ModelCheckpoint(ckpt, monitor="val_loss",
                                        save_best_only=True, verbose=0),
    ]

    history = model.fit(
        train_batcher,
        validation_data=val_batcher,
        epochs=epochs,
        callbacks=callbacks,
        verbose=1,
    )
    print(f"\n{cell_type}: best val_loss = {min(history.history['val_loss']):.6f}")
    return model, history

### Scale the returns for the regression head

`y_*.npy` holds raw percentage returns (std ~0.2% at 15m). Regressing directly
on values that small makes MSE gradients tiny, so the head is trained against
the **target-scaler-normalised** returns and the predictions are inverted back
to percent before any metric is computed.

The scaler was fitted on the training split locally - it is loaded, never refit.

In [ ]:
MEAN_T = target_scaler.mean_.astype(np.float32)
SCALE_T = target_scaler.scale_.astype(np.float32)
print("target scaler means :", np.round(MEAN_T, 5))
print("target scaler scales:", np.round(SCALE_T, 5))


def to_scaled(y):
    return ((y - MEAN_T) / SCALE_T).astype(np.float32)


def from_scaled(y):
    return (y * SCALE_T + MEAN_T).astype(np.float32)

In [ ]:
# --- GRU (primary architecture) -------------------------------------------
gru_model, gru_hist = train_recurrent("GRU", tag="gru")

In [ ]:
# --- LSTM (reference) ------------------------------------------------------
lstm_model, lstm_hist = train_recurrent("LSTM", tag="lstm")

### Pick the architecture on validation, not on test

Choose whichever recurrent model has the lower validation loss / better
validation macro-F1. Only then evaluate the winner on the test split.

In [ ]:
def recurrent_predict(model, batcher, n):
    raw = model.predict(batcher, verbose=0)
    returns = from_scaled(raw["return_head"][:n])
    direction = raw["direction_head"][:n].argmax(axis=-1).astype(np.int64)
    return returns, direction


# y_*.npy already holds exactly one row per sample - no LOOKBACK offset here.
n_val = len(y_val)
n_test = len(y_test)

gru_val_pred, gru_val_dir = recurrent_predict(gru_model, val_batcher, n_val)
lstm_val_pred, lstm_val_dir = recurrent_predict(lstm_model, val_batcher, n_val)

print("GRU  - VALIDATION"); display(regression_metrics(y_val, gru_val_pred))
print("LSTM - VALIDATION"); display(regression_metrics(y_val, lstm_val_pred))

print("GRU  direction - VALIDATION")
display(direction_metrics(d_val, gru_val_dir))
print("LSTM direction - VALIDATION")
display(direction_metrics(d_val, lstm_val_dir))


In [ ]:
# Validation-based model selection.
gru_f1 = direction_metrics(d_val, gru_val_dir)["macro_F1"].mean()
lstm_f1 = direction_metrics(d_val, lstm_val_dir)["macro_F1"].mean()

print(f"mean macro-F1 (validation):  GRU={gru_f1:.4f}   LSTM={lstm_f1:.4f}")
BEST = "GRU" if gru_f1 >= lstm_f1 else "LSTM"
print("selected on validation:", BEST)
print("\nThis is a hypothesis, not a conclusion - rerun with different seeds "
      "before reading anything into a small difference.")


## 8. Test-set evaluation (touched exactly once)

Every number below comes from genuinely unseen future data. The test split is
separated from training by a purge/embargo of `PURGE_SAMPLES` samples
(`lookback + max horizon`), so no training label is resolved from any candle
inside the test feature windows - verified locally by `src/audit_leakage.py`.


In [ ]:
best_model = gru_model if BEST == "GRU" else lstm_model
best_val_pred, best_val_dir = ((gru_val_pred, gru_val_dir) if BEST == "GRU"
                               else (lstm_val_pred, lstm_val_dir))
best_test_pred, best_test_dir = recurrent_predict(best_model, test_batcher, n_test)

print(f"=== {BEST} - TEST REGRESSION ===")
test_reg = regression_metrics(y_test, best_test_pred)
display(test_reg)

print(f"=== {BEST} - TEST DIRECTION ===")
test_dir = direction_metrics(d_test, best_test_dir)
display(test_dir)


In [ ]:
# Side-by-side against the naive floors - the only comparison that matters.
print("MAE (% return) - test")
compare = pd.DataFrame({
    "naive_zero": base_reg["MAE_%"],
    f"{BEST}_recurrent": test_reg["MAE_%"],
    "xgboost": regression_metrics(y_test, xgb_test_pred)["MAE_%"],
})
display(compare)

print("directional accuracy - test")
base_acc = direction_metrics(d_test, base_dir_pred)["accuracy"]
display(pd.DataFrame({
    "majority_class": base_acc,
    f"{BEST}_recurrent": test_dir["accuracy"],
    "xgboost": direction_metrics(d_test, xgb_test_dir)["accuracy"],
}))

In [ ]:
# Confusion matrix, one per horizon - never a single pooled matrix.
for i, h in enumerate(HORIZONS):
    print(f"--- {h} ---")
    display(show_confusion(d_test, best_test_dir, horizon_index=i))


## 9. Save the trained models

Everything needed for inference: the model, the scaler, the target scaler, the
thresholds and the feature order.

In [ ]:
OUT = DATA_DIR / "trained"
OUT.mkdir(parents=True, exist_ok=True)

best_model.save(OUT / f"{BEST.lower()}_multitask.keras")

with open(OUT / "inference_bundle.json", "w", encoding="utf-8") as fh:
    json.dump({
        "architecture": BEST,
        "lookback": LOOKBACK,
        "feature_columns": META["feature_columns"],
        "horizons": HORIZONS,
        "direction_classes": META["direction_classes"],
        "thresholds": META["thresholds"]["per_horizon"],
        "target_scaler_mean": MEAN_T.tolist(),
        "target_scaler_scale": SCALE_T.tolist(),
        "val_macro_f1": float(max(gru_f1, lstm_f1)),
    }, fh, indent=2)

print("saved to", OUT)

## 10. Backtest scaffold (deliberately empty)

**No backtest results are produced or claimed here.** Directional accuracy is
not profitability: a model can be 55% accurate and still lose money after fees,
slippage and bad sizing.

The honest sequence is: *train -> evaluate -> backtest -> paper trade*, and only
the first two happen in this notebook. When you do build the backtest, it must
model:

* **fees** - taker fee on entry *and* exit (~0.04-0.10% per side on Binance)
* **slippage** - the next candle's open, not the close you predicted on
* **latency** - the signal is only actionable after the candle closes
* **confidence gate** - skip trades below a minimum predicted-probability
* **no-trade zone** - honour the NEUTRAL band instead of forcing a position
* **purge** - never let a position span the train/test boundary

In [ ]:
# ---------------------------------------------------------------------------
# BACKTEST - intentionally not implemented. Sketch only.
# ---------------------------------------------------------------------------
# probs = best_model.predict(test_batcher)["direction_head"][:, :, :]  # (n, 6, 3)
#
# for i, h in enumerate(HORIZONS):
#     conf = probs[:, i, :].max(axis=1)
#     signal = np.where(conf < MIN_CONFIDENCE, 1,          # 1 == NEUTRAL == flat
#                       probs[:, i, :].argmax(axis=1))
#     # enter at the OPEN of the candle after the signal, exit after h candles
#     # subtract fees on both legs and a slippage estimate per leg
#     # then report equity curve, Sharpe, max drawdown, hit rate, turnover
#
# Report nothing until this is implemented and run on truly unseen data.
print("Backtest deliberately not implemented - see the markdown above.")

## 11. Honest limitations

* **Directional accuracy near 33% is what a three-class coin flip looks like.**
  Anything meaningfully above that on the *test* split is interesting; anything
  near it is noise.
* **Financial returns are close to unpredictable at 15-minute horizons.** A
  model that beats the naive baseline slightly, consistently, is a good result.
  Expect small edges, not large ones.
* **A single train/test split is one sample.** Re-run with several seeds and
  consider walk-forward validation before believing any number.
* **Regime change is real.** A model fitted on 2017-2024 may behave differently
  in 2026. The chronological split protects the *evaluation*, not the model's
  future behaviour.
* **No profitability is claimed anywhere.** Accuracy is not profit; fees and
  slippage dominate short-horizon strategies.